# Renderizar Motion Graphics (HyperFrames) en Colab (sin apt, via pip)

Esta version usa pip para FFmpeg (imageio-ffmpeg) y Chromium (Playwright), y nvm para Node.js.
Incluye instalacion de librerias del sistema necesarias para Chrome en Ubuntu 24.04.

Pasos:
1. Ejecuta las celdas en orden.
2. Sube un .zip de tu proyecto (con index.html).
3. Descarga el output.mp4.


In [ ]:
# 1) Node.js 22 via nvm (sin apt)
!curl -o- https://raw.githubusercontent.com/nvm-sh/nvm/v0.39.7/install.sh | bash
!export NVM_DIR="$HOME/.nvm" && . "$NVM_DIR/nvm.sh" && nvm install 22 && nvm use 22 && node --version && npm --version


In [ ]:
# 2) FFmpeg y Chromium via pip
!pip install -q imageio-ffmpeg playwright
!playwright install chromium
import imageio_ffmpeg, playwright
print("ffmpeg bin:", imageio_ffmpeg.get_ffmpeg_exe())


In [ ]:
# 2b) Librerias del sistema que necesita Chrome (headless) en Ubuntu 24.04
!apt-get update -qq
!apt-get install -y -qq libnss3 libnspr4 libatk1.0-0 libatk-bridge2.0-0 libcups2 libdrm2 libxkbcommon0 libatspi2.0-0 libxcomposite1 libxdamage1 libxfixes3 libxrandr2 libgbm1 libpango-1.0-0 libcairo2 libasound2t64 libxshmfence1


In [ ]:
# 3) Configurar variables de entorno para HyperFrames
import os, glob
ffmpeg_bin = imageio_ffmpeg.get_ffmpeg_exe()
os.environ['HYPERFRAMES_FFMPEG_PATH'] = ffmpeg_bin
chromium_candidates = glob.glob('/root/.cache/ms-playwright/chromium-*/chrome-linux/chrome') + glob.glob('/home/*/.cache/ms-playwright/chromium-*/chrome-linux/chrome')
system_chromium = None
for c in ['/usr/bin/chromium','/usr/bin/chromium-browser','/usr/bin/google-chrome']:
    if os.path.exists(c):
        system_chromium = c
        break
chromium_bin = chromium_candidates[0] if chromium_candidates else system_chromium
if chromium_bin:
    os.environ['CHROME_PATH'] = chromium_bin
    os.environ['HYPERFRAMES_BROWSER_PATH'] = chromium_bin
    os.environ['PUPPETEER_EXECUTABLE_PATH'] = chromium_bin
print('FFMPEG:', os.environ.get('HYPERFRAMES_FFMPEG_PATH'))
print('CHROME:', os.environ.get('CHROME_PATH'))
print('HYPERFRAMES_BROWSER_PATH:', os.environ.get('HYPERFRAMES_BROWSER_PATH'))


In [ ]:
# 4) Subir el proyecto (zip)
from google.colab import files
uploaded = files.upload()
print("Subido:", list(uploaded.keys()))


In [ ]:
# 5) Descomprimir
import zipfile, os
zip_name = list(uploaded.keys())[0]
os.makedirs('/content/project', exist_ok=True)
with zipfile.ZipFile(zip_name, 'r') as z:
    z.extractall('/content/project')
entries = os.listdir('/content/project')
if len(entries) == 1 and os.path.isdir(os.path.join('/content/project', entries[0])):
    inner = os.path.join('/content/project', entries[0])
    for f in os.listdir(inner):
        os.rename(os.path.join(inner, f), os.path.join('/content/project', f))
    os.rmdir(inner)
print('Contenido:', os.listdir('/content/project'))


In [ ]:
# 5b) Instalar HyperFrames globalmente (evita prompt de npx)
import os
nvm_dir = os.path.expanduser("~/.nvm")
install_cmd = 'export NVM_DIR="' + nvm_dir + '" && . "$NVM_DIR/nvm.sh" && nvm use 22 && npm install -g hyperframes'
!{install_cmd}


In [ ]:
# 6) Renderizar a MP4 (binario global)
import os
nvm_dir = os.path.expanduser("~/.nvm")
cmd = 'export NVM_DIR="' + nvm_dir + '" && . "$NVM_DIR/nvm.sh" && nvm use 22 && cd /content/project && hyperframes render --output /content/output.mp4'
!{cmd}


In [ ]:
# 7) Descargar el video
from google.colab import files
files.download("/content/output.mp4")
